# House Prices: Iowa Survival Model v4
**アイオワの冬における資産価値を多角的に分析し、Generational Adaptive Ensemble で予測する**

| Paradigm | Model | Perspective |
|----------|-------|-------------|
| Linear | Ridge, ElasticNet | 線形関係 (RobustScaler + Guard clip) |
| Boosting | LightGBM, CatBoost | 木ベースの閾値分岐、交互作用 |
| NN | TabNet | Attentionベースの特徴量選択 |

**Iowa Features (v3 base)**: Winter_Resilience, Thermal_Efficiency, Has_Basement, Bsmt_Liveability
**V4 Generational**: Young_Transient, Mid_Productive, Senior_Settled, Snow_Maintenance_Burden, Senior_Roof_Penalty, Quiet_Density_Index, Life_Line_Proximity, Functional_Survival_Qual

**Pipeline**: Multi-Seed (5seeds) → 2-Layer Stacking (HuberRegressor) → Generational Adaptive Ensemble → Kaggle Submit

## Setup

In [ ]:
import sys, shutil, os, json, zipfile
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
HAS_GPU = shutil.which('nvidia-smi') is not None

# --- Colab runtime health check ---
if IS_COLAB:
    try:
        import subprocess
        _hc = subprocess.run(['python3', '-c', 'print("ok")'],
                             capture_output=True, text=True, timeout=5)
        if _hc.returncode != 0:
            raise RuntimeError("Runtime subprocess failed")
        print("Colab runtime: OK")
    except Exception as e:
        print(f"ERROR: Colab runtime unhealthy: {e}")
        print("  -> Colabセッションが切断されている可能性があります")
        print("  -> VSCode: Ctrl+Shift+P → 'Jupyter: Clear All Jupyter Remote Server Connections'")
        print("  -> Colabでランタイムを再起動してから再接続してください")
        raise RuntimeError("Colab runtime not available") from e

print(f"Environment: {'Colab' if IS_COLAB else 'Local'}, GPU: {HAS_GPU}")
print(f"CWD: {os.getcwd()}")

if HAS_GPU:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    for line in result.stdout.split('\n')[:4]:
        print(line)

if IS_COLAB:
    !pip install -q pytorch-tabnet lightgbm xgboost catboost shap
else:
    import importlib
    for pkg in ['pytorch_tabnet', 'xgboost', 'catboost', 'shap']:
        if importlib.util.find_spec(pkg) is None:
            !pip install {pkg.replace('_', '-')}

In [ ]:
if IS_COLAB:
    import os
    if not os.path.exists('/content/drive/MyDrive'):
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except Exception as e:
            print(f"drive.mount() skipped: {e}")
            print("  (VSCode remote connection does not support interactive auth)")
            print("  Data will be loaded via Kaggle CLI or local copy instead.")

## Imports

In [ ]:
import numpy as np
import pandas as pd
import torch
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error
from pytorch_tabnet.tab_model import TabNetRegressor
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch device: {DEVICE}")

In [ ]:
import os, json, zipfile
from pathlib import Path

# --- Resolve DATA_DIR: local/existing data first ---
_candidates = []
if IS_COLAB:
    _candidates += [
        Path("data"),                                         # CWD/data (previous run)
        Path("/content/data"),                                # absolute
        Path("/content/drive/MyDrive/kaggle-data/house-prices"),  # Google Drive
    ]
else:
    try:
        nb_dir = Path(__vsc_ipynb_file__).resolve().parent
        _candidates.insert(0, nb_dir / "data")
    except (NameError, OSError):
        pass
    _candidates += [Path("data"), Path("house-prices/data")]

DATA_DIR = None
for candidate in _candidates:
    if (candidate / "train.csv").exists():
        DATA_DIR = candidate
        print(f"Data found at: {candidate}")
        break

if DATA_DIR is None:
    DATA_DIR = Path("data") if IS_COLAB else Path("house-prices/data")
    print(f"No existing data found, will download to: {DATA_DIR}")

DATA_DIR.mkdir(parents=True, exist_ok=True)

# --- Download data if needed (Drive copy > Kaggle CLI) ---
if not (DATA_DIR / "train.csv").exists():
    # Method 1: Copy from Google Drive (if mounted)
    if IS_COLAB:
        drive_data = Path("/content/drive/MyDrive/kaggle-data/house-prices")
        if drive_data.exists() and (drive_data / "train.csv").exists():
            print("Copying data from Google Drive...")
            for f in ["train.csv", "test.csv", "data_description.txt", "sample_submission.csv"]:
                src = drive_data / f
                if src.exists():
                    shutil.copy2(str(src), str(DATA_DIR / f))
            print("Done.")

    # Method 2: Kaggle CLI (fallback)
    if not (DATA_DIR / "train.csv").exists():
        kaggle_dir = Path.home() / ".kaggle"
        kaggle_json_path = kaggle_dir / "kaggle.json"

        if not kaggle_json_path.exists():
            creds = None
            # Try Colab Secrets
            try:
                from google.colab import userdata
                creds = {'username': userdata.get('KAGGLE_USERNAME'),
                         'key': userdata.get('KAGGLE_KEY')}
            except Exception:
                pass
            # Try Drive kaggle.json
            if (not creds or not creds.get('key')):
                drive_kaggle = Path("/content/drive/MyDrive/.kaggle/kaggle.json")
                if drive_kaggle.exists():
                    creds = json.loads(drive_kaggle.read_text())
                    print("Loaded kaggle.json from Google Drive")
            if not creds or not creds.get('key'):
                raise FileNotFoundError(
                    "kaggle.json not found. Options:\n"
                    "  1. Upload train.csv & test.csv to /content/data/\n"
                    "  2. Set KAGGLE_USERNAME/KAGGLE_KEY in Colab Secrets\n"
                    "  3. Place kaggle.json at Google Drive: MyDrive/.kaggle/kaggle.json")
            kaggle_dir.mkdir(exist_ok=True)
            kaggle_json_path.write_text(json.dumps(creds))
            os.chmod(str(kaggle_json_path), 0o600)

        print("Downloading via kaggle CLI...")
        !kaggle competitions download -c house-prices-advanced-regression-techniques -p {DATA_DIR}

        zip_path = DATA_DIR / "house-prices-advanced-regression-techniques.zip"
        if zip_path.exists():
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(DATA_DIR)
            zip_path.unlink()
        elif not (DATA_DIR / "train.csv").exists():
            raise FileNotFoundError(
                "Download failed. Easiest fix: upload train.csv & test.csv to /content/data/")

TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"
print(f"Train: {TRAIN_CSV.exists()}, Test: {TEST_CSV.exists()}")

# Output directories
if IS_COLAB:
    SUBMISSION_DIR = '/content/submissions'
    FIGURES_DIR = '/content/figures'
else:
    _project_root = Path(DATA_DIR).resolve().parent if Path(DATA_DIR).name == 'data' else Path('.')
    SUBMISSION_DIR = str(_project_root / 'submissions')
    FIGURES_DIR = str(_project_root / 'figures')

os.makedirs(SUBMISSION_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

## Data Loading

In [58]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)

# Outlier removal (GrLivArea > 4000 & SalePrice < 300000)
outlier_mask = (train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)
print(f"Outliers removed: {outlier_mask.sum()}")
train = train[~outlier_mask].reset_index(drop=True)

y = np.log1p(train['SalePrice'])
X = train.drop(columns=['SalePrice'])
X_test = test.copy()
test_ids = test['Id']

print(f"Train: {X.shape}, Test: {X_test.shape}")

Outliers removed: 2
Train: (1458, 80), Test: (1459, 80)


## Constants

In [59]:
CLIP_MIN = 35000
CLIP_MAX = 600000

NEIGHBORHOOD_COORDS = {
    "Blmngtn": (42.062736, -93.641598), "Blueste": (42.055300, -93.648000),
    "BrDale": (42.052702, -93.627195), "BrkSide": (42.033590, -93.627692),
    "ClearCr": (42.060438, -93.621175), "CollgCr": (42.021678, -93.685793),
    "Crawfor": (42.015962, -93.640486), "Edwards": (42.022110, -93.663703),
    "Gilbert": (42.063379, -93.647255), "Greens": (42.063150, -93.642700),
    "GrnHill": (42.061000, -93.643500), "IDOTRR": (42.022632, -93.619070),
    "Landmrk": (42.030000, -93.620000), "MeadowV": (42.032428, -93.641709),
    "Mitchel": (42.033900, -93.614700), "NAmes": (42.044800, -93.650500),
    "NoRidge": (42.050760, -93.656810), "NPkVill": (42.050000, -93.625000),
    "NridgHt": (42.060200, -93.656200), "NWAmes": (42.051000, -93.659100),
    "OldTown": (42.028270, -93.613600), "SWISU": (42.017200, -93.651600),
    "Sawyer": (42.035050, -93.666120), "SawyerW": (42.035300, -93.685700),
    "Somerst": (42.052370, -93.645830), "StoneBr": (42.060000, -93.648500),
    "Timber": (42.017000, -93.681000), "Veenker": (42.040800, -93.651700),
}
AMES_CENTER = (42.05, -93.65)
HIGH_PRICE_CENTER = (42.057, -93.646)

REDUNDANT_COLS = [
    'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'GrLivArea',
    'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath',
    'YearBuilt', 'GarageArea'
]

## Preprocessor (pipeline.ipynb 共通)

In [60]:
class HousePricesPreprocessor(BaseEstimator, TransformerMixin):
    """Shared preprocessor from pipeline.ipynb"""

    NA_MEANS_NONE = [
        'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
        'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu',
        'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
        'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType'
    ]

    ORDINAL_MAP = {
        'ExterQual': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'ExterCond': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'BsmtQual': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'BsmtCond': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'BsmtExposure': {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4},
        'HeatingQC': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'KitchenQual': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'FireplaceQu': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'GarageQual': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'GarageCond': {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
        'GarageFinish': {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3},
        'PoolQC': {'None': 0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex': 4},
        'Fence': {'None': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4},
        'BsmtFinType1': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
        'BsmtFinType2': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
        'Functional': {'Sal': 1, 'Sev': 2, 'Maj2': 3, 'Maj1': 4, 'Mod': 5, 'Min2': 6, 'Min1': 7, 'Typ': 8},
        'LandSlope': {'Sev': 1, 'Mod': 2, 'Gtl': 3},
        'PavedDrive': {'N': 0, 'P': 1, 'Y': 2},
        'CentralAir': {'N': 0, 'Y': 1},
        'Street': {'Grvl': 0, 'Pave': 1},
        'Utilities': {'ELO': 0, 'NoSeWa': 1, 'NoSewr': 2, 'AllPub': 3},
    }

    def __init__(self, selected_features=None):
        self.selected_features = selected_features

    def _fill_missing(self, df):
        for col in self.NA_MEANS_NONE:
            if col in df.columns:
                df[col] = df[col].fillna('None')
        num_cols = df.select_dtypes(include=[np.number]).columns
        for col in num_cols:
            if df[col].isnull().any():
                if col in ['LotFrontage']:
                    df[col] = df[col].fillna(df[col].median())
                else:
                    df[col] = df[col].fillna(0)
        obj_cols = df.select_dtypes(include=['object']).columns
        for col in obj_cols:
            if df[col].isnull().any():
                df[col] = df[col].fillna(df[col].mode()[0] if len(df[col].mode()) > 0 else 'None')
        return df

    def _ordinal_encode(self, df):
        for col, mapping in self.ORDINAL_MAP.items():
            if col in df.columns:
                df[col] = df[col].map(mapping).fillna(0).astype(int)
        return df

    def _add_geo_features(self, df):
        if 'Neighborhood' in df.columns:
            df['Latitude'] = df['Neighborhood'].map({k: v[0] for k, v in NEIGHBORHOOD_COORDS.items()})
            df['Longitude'] = df['Neighborhood'].map({k: v[1] for k, v in NEIGHBORHOOD_COORDS.items()})
            df['Latitude'] = df['Latitude'].fillna(AMES_CENTER[0])
            df['Longitude'] = df['Longitude'].fillna(AMES_CENTER[1])
            df['Dist_to_Center'] = np.sqrt(
                (df['Latitude'] - AMES_CENTER[0])**2 + (df['Longitude'] - AMES_CENTER[1])**2)
            df['Dist_to_HighPrice_Center'] = np.sqrt(
                (df['Latitude'] - HIGH_PRICE_CENTER[0])**2 + (df['Longitude'] - HIGH_PRICE_CENTER[1])**2)
        return df

    def _add_domain_features(self, df):
        df['TotalSF'] = df.get('TotalBsmtSF', 0) + df.get('1stFlrSF', 0) + df.get('2ndFlrSF', 0)
        if 'TotalBsmtSF' in df.columns:
            df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
        df['TotalBath'] = (df.get('FullBath', 0) + df.get('HalfBath', 0)
                          + df.get('BsmtFullBath', 0) + df.get('BsmtHalfBath', 0))
        if 'FullBath' in df.columns:
            df['TotalBath'] = df['FullBath'] + df['HalfBath'] + df['BsmtFullBath'] + df['BsmtHalfBath']
        df['HouseAge'] = df['YrSold'] - df.get('YearBuilt', df['YrSold'])
        if 'YearBuilt' in df.columns:
            df['HouseAge'] = df['YrSold'] - df['YearBuilt']
        df['RemodAge'] = df['YrSold'] - df.get('YearRemodAdd', df['YrSold'])
        if 'YearRemodAdd' in df.columns:
            df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']

        porch_cols = ['OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch']
        df['TotalPorchSF'] = sum(df.get(c, 0) for c in porch_cols)

        total_sf = df.get('TotalSF', 1)
        gr_liv = df.get('GrLivArea', df.get('TotalSF', 1))
        df['Living_Space_Ratio'] = np.where(total_sf > 0, gr_liv / total_sf, 1.0)

        bsmt_sf = df.get('TotalBsmtSF', 0)
        second_sf = df.get('2ndFlrSF', 0)
        df['Luxury_Space_Index'] = np.where(
            total_sf > 0,
            (bsmt_sf * 0.3 + second_sf * 0.4) / total_sf, 0.0)

        df['IsNew'] = (df['HouseAge'] == 0).astype(int)
        df['HasRemod'] = (df['RemodAge'] >= 0).astype(int)
        df['HasGarage'] = (df.get('GarageCars', 0) > 0).astype(int)
        df['HasBsmt'] = (df.get('TotalBsmtSF', 0) > 0).astype(int)
        df['Has2ndFlr'] = (df.get('2ndFlrSF', 0) > 0).astype(int)
        df['HasPool'] = (df.get('PoolArea', 0) > 0).astype(int)

        # QOL Score
        qual_cols = ['OverallQual', 'ExterQual', 'KitchenQual']
        cond_cols = ['OverallCond', 'ExterCond']
        q_sum = sum(df.get(c, 0) for c in qual_cols)
        c_sum = sum(df.get(c, 0) for c in cond_cols)
        func_val = df.get('Functional', 8)
        df['QOL_Score'] = q_sum * 0.5 + c_sum * 0.3 + func_val * 0.2

        return df

    def _drop_redundant(self, df):
        drop_cols = [c for c in REDUNDANT_COLS if c in df.columns]
        df = df.drop(columns=drop_cols)
        df = df.drop(columns=['Id'], errors='ignore')
        return df

    def _fix_skewness(self, df):
        num_cols = df.select_dtypes(include=[np.number]).columns
        for col in num_cols:
            if df[col].nunique() > 2 and df[col].skew() > 0.7:
                df[col] = np.log1p(df[col].clip(lower=0))
        return df

    def fit_transform(self, X, y=None):
        df = X.copy()
        df = self._fill_missing(df)
        df = self._ordinal_encode(df)
        df = self._add_geo_features(df)
        df = self._add_domain_features(df)
        df = self._drop_redundant(df)
        df = self._fix_skewness(df)

        # Label encode remaining categoricals
        self.label_encoders_ = {}
        self.cat_columns_ = []
        for col in df.select_dtypes(include=['object']).columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            self.label_encoders_[col] = le
            self.cat_columns_.append(col)

        if self.selected_features is not None:
            keep = [c for c in self.selected_features if c in df.columns]
            df = df[keep]

        self.feature_names_ = list(df.columns)
        self.cat_feature_indices_ = [i for i, c in enumerate(self.feature_names_) if c in self.cat_columns_]
        return df

    def transform(self, X):
        df = X.copy()
        df = self._fill_missing(df)
        df = self._ordinal_encode(df)
        df = self._add_geo_features(df)
        df = self._add_domain_features(df)
        df = self._drop_redundant(df)
        df = self._fix_skewness(df)

        for col, le in self.label_encoders_.items():
            if col in df.columns:
                # Handle unseen labels
                known = set(le.classes_)
                df[col] = df[col].astype(str).apply(lambda x: x if x in known else le.classes_[0])
                df[col] = le.transform(df[col])

        if self.selected_features is not None:
            keep = [c for c in self.selected_features if c in df.columns]
            df = df[keep]

        return df

## KNN Geo Price Feature

In [61]:
from sklearn.neighbors import KNeighborsRegressor

def compute_knn_geo_price(X, y, X_test, n_neighbors=10, n_splits=5, seed=42):
    """Leak-free KNN geographic price feature."""
    def _get_coords(df):
        lat = df['Neighborhood'].map({k: v[0] for k, v in NEIGHBORHOOD_COORDS.items()}).fillna(AMES_CENTER[0])
        lon = df['Neighborhood'].map({k: v[1] for k, v in NEIGHBORHOOD_COORDS.items()}).fillna(AMES_CENTER[1])
        return np.column_stack([lat.values, lon.values])

    coords_train = _get_coords(X)
    coords_test = _get_coords(X_test)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))

    for train_idx, val_idx in kf.split(X):
        knn = KNeighborsRegressor(n_neighbors=n_neighbors)
        knn.fit(coords_train[train_idx], y.values[train_idx])
        oof[val_idx] = knn.predict(coords_train[val_idx])
        test_preds += knn.predict(coords_test) / n_splits

    return oof, test_preds

knn_train, knn_test = compute_knn_geo_price(X, y, X_test)
X['KNN_Geo_Price'] = knn_train
X_test['KNN_Geo_Price'] = knn_test
print(f"KNN Geo Price: train mean={knn_train.mean():.4f}, test mean={knn_test.mean():.4f}")

KNN Geo Price: train mean=12.0382, test mean=12.0293


## Feature Selection (LightGBM importance)

In [62]:
import lightgbm as lgb

SHAP_TOP_N = 40

# Preprocess all features for importance ranking
_prep_fs = HousePricesPreprocessor(selected_features=None)
_X_fs = _prep_fs.fit_transform(X, y)

_lgb_fs = lgb.LGBMRegressor(
    n_estimators=2000, learning_rate=0.01, max_depth=3,
    num_leaves=7, min_child_samples=50, subsample=0.8,
    colsample_bytree=0.8, random_state=42, verbose=-1
)
_lgb_fs.fit(_X_fs.values.astype(np.float64), y)

_importances = pd.Series(_lgb_fs.feature_importances_, index=_prep_fs.feature_names_)
_importances = _importances.sort_values(ascending=False)
_importances = _importances.drop(labels=[c for c in _importances.index if c.endswith('_te')], errors='ignore')

SELECTED_FEATURES = _importances.head(SHAP_TOP_N).index.tolist()
print(f"Selected {len(SELECTED_FEATURES)} features")
print(SELECTED_FEATURES[:10], '...')

Selected 40 features
['TotalSF', 'LotArea', 'KNN_Geo_Price', 'HouseAge', 'GarageYrBlt', 'QOL_Score', 'BsmtUnfSF', 'Living_Space_Ratio', 'OverallCond', 'BsmtFinSF1'] ...


## Pipeline Wrappers

In [ ]:
class CatBoostPipeline:
    """CatBoost: cat_features auto-passing"""
    def __init__(self, model, selected_features=None):
        self.preprocess = HousePricesPreprocessor(selected_features=selected_features)
        self.model = model
    def fit(self, X, y):
        X_t = self.preprocess.fit_transform(X, y)
        self.model.fit(X_t, y, cat_features=self.preprocess.cat_feature_indices_)
        return self
    def predict(self, X):
        return self.model.predict(self.preprocess.transform(X))

class SklearnPipeline:
    """sklearn: LabelEncode remaining objects + optional scaling"""
    def __init__(self, model, selected_features=None, scale=False):
        self.preprocess = HousePricesPreprocessor(selected_features=selected_features)
        self.model = model
        self.label_maps_ = {}
        self.scale = scale
        self.scaler_ = None
    def fit(self, X, y):
        X_t = self.preprocess.fit_transform(X, y)
        X_t = self._le_fit(X_t)
        X_val = X_t.values.astype(np.float64)
        if self.scale:
            self.scaler_ = StandardScaler()
            X_val = self.scaler_.fit_transform(X_val)
        self.model.fit(X_val, y)
        return self
    def predict(self, X):
        X_t = self.preprocess.transform(X)
        X_t = self._le_transform(X_t)
        X_val = X_t.values.astype(np.float64)
        if self.scaler_ is not None:
            X_val = self.scaler_.transform(X_val)
        return self.model.predict(X_val)
    def _le_fit(self, df):
        self.label_maps_ = {}
        for col in df.select_dtypes(include=['object']).columns:
            vals = sorted(df[col].unique())
            self.label_maps_[col] = {v: i for i, v in enumerate(vals)}
            df[col] = df[col].map(self.label_maps_[col]).fillna(-1).astype(int)
        return df
    def _le_transform(self, df):
        for col, m in self.label_maps_.items():
            if col in df.columns:
                df[col] = df[col].map(m).fillna(-1).astype(int)
        return df

SEED = 42

## TabNet CV Harness

In [ ]:
import copy

N_SPLITS = 5

def run_tabnet_cv(X_arr, y_arr, feature_names, cat_idxs, cat_dims, n_splits=N_SPLITS):
    """CV for TabNet (lightweight: reduced epochs)."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    num_idxs = [i for i in range(X_arr.shape[1]) if i not in cat_idxs]
    oof = np.zeros(len(X_arr))
    fold_scores = []
    last_model = None

    tabnet_params = dict(
        n_d=16, n_a=16, n_steps=4, gamma=1.5, lambda_sparse=1e-3,
        cat_idxs=cat_idxs, cat_dims=cat_dims, cat_emb_dim=1,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2, weight_decay=1e-5),
        scheduler_fn=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts,
        scheduler_params=dict(T_0=50, T_mult=2, eta_min=1e-5),
        mask_type='entmax', seed=SEED, verbose=0, device_name=DEVICE)
    fit_params = dict(max_epochs=200, patience=20, batch_size=64,
                      virtual_batch_size=32, num_workers=0, drop_last=False)

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_arr)):
        scaler = StandardScaler()
        X_tr = X_arr[tr_idx].copy()
        X_va = X_arr[va_idx].copy()
        X_tr[:, num_idxs] = scaler.fit_transform(X_tr[:, num_idxs])
        X_va[:, num_idxs] = scaler.transform(X_va[:, num_idxs])
        y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]

        model = TabNetRegressor(**{**tabnet_params, 'seed': SEED + fold})
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
                  eval_metric=['rmse'], **fit_params)

        preds = model.predict(X_va).flatten()
        oof[va_idx] = preds
        rmse = np.sqrt(mean_squared_error(y_va.flatten(), preds))
        fold_scores.append(rmse)
        last_model = model
        import gc; gc.collect()

    cv_mean = np.mean(fold_scores)
    cv_std = np.std(fold_scores)
    oof_rmse = np.sqrt(mean_squared_error(y_arr.flatten(), oof))
    print(f"  {'TabNet':16s} [{'NN':8s}]  OOF={oof_rmse:.5f}  CV={cv_mean:.5f} +/- {cv_std:.5f}")
    return {'oof': oof, 'fold_scores': fold_scores, 'cv_mean': cv_mean,
            'cv_std': cv_std, 'oof_rmse': oof_rmse, 'paradigm': 'NN',
            'model': last_model}

## Iowa Survival Features (v3 base + v4 generational)
- v3: Winter_Resilience, Thermal_Efficiency, Has_Basement, Bsmt_Liveability
- v4: Generational Segment, Snow Maintenance, Social Resilience, Functional Survival

In [ ]:
from sklearn.preprocessing import RobustScaler
import gc, copy

# ============================================================
# Iowa Survival Features (v3 base)
# ============================================================
def add_iowa_survival_features(df):
    """
    Generate 'survival & comfort' features for Iowa winters.
    Called BEFORE ordinal encoding (raw string values available).
    """
    heating_map = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
    exposure_map = {'None': 0, 'NA': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}

    bsmt_fin = df.get('BsmtFinSF1', pd.Series(0, index=df.index)).fillna(0)

    if df.get('HeatingQC') is not None and df['HeatingQC'].dtype == object:
        heat = df['HeatingQC'].map(heating_map).fillna(3)
    else:
        heat = df.get('HeatingQC', pd.Series(3, index=df.index)).fillna(3)

    if df.get('BsmtExposure') is not None and df['BsmtExposure'].dtype == object:
        expo = df['BsmtExposure'].fillna('None').map(exposure_map).fillna(0)
    else:
        expo = df.get('BsmtExposure', pd.Series(0, index=df.index)).fillna(0)

    if df.get('CentralAir') is not None and df['CentralAir'].dtype == object:
        central_air = (df['CentralAir'] == 'Y').astype(int)
    else:
        central_air = df.get('CentralAir', pd.Series(1, index=df.index)).fillna(1)

    bsmt_max = bsmt_fin.max()
    bsmt_norm = bsmt_fin / (bsmt_max + 1e-9) if bsmt_max > 0 else bsmt_fin
    heat_norm = heat / 5.0
    expo_norm = expo / 4.0
    df['Bsmt_Liveability'] = bsmt_norm * heat_norm * expo_norm

    total_bsmt = df.get('TotalBsmtSF', bsmt_fin).fillna(0)
    df['Has_Basement'] = (total_bsmt > 0).astype(int)
    no_bsmt = df['Has_Basement'] == 0
    df.loc[no_bsmt, 'Bsmt_Liveability'] = 0.0

    df['Bsmt_Liveability_sq'] = df['Bsmt_Liveability'] ** 2
    df['Bsmt_Liveability_log'] = np.log1p(df['Bsmt_Liveability'] * 100)
    df.loc[no_bsmt, 'Bsmt_Liveability_sq'] = 0.0
    df.loc[no_bsmt, 'Bsmt_Liveability_log'] = 0.0

    df['Winter_Resilience'] = heat_norm * central_air * df['Bsmt_Liveability']
    df.loc[no_bsmt, 'Winter_Resilience'] = 0.0

    total_sf = (df.get('TotalBsmtSF', pd.Series(0, index=df.index)).fillna(0)
                + df.get('1stFlrSF', pd.Series(0, index=df.index)).fillna(0)
                + df.get('2ndFlrSF', pd.Series(0, index=df.index)).fillna(0))
    df['Thermal_Efficiency'] = np.where(
        total_sf > 0,
        df['Winter_Resilience'] / (total_sf / 1000.0 + 1e-9),
        0.0
    )

    if 'KNN_Geo_Price' in df.columns:
        df['Winter_x_Geo'] = df['Winter_Resilience'] * df['KNN_Geo_Price']
        df['Bsmt_Live_x_Geo'] = df['Bsmt_Liveability'] * df['KNN_Geo_Price']

    return df

IOWA_FEATURES = [
    'Bsmt_Liveability', 'Bsmt_Liveability_sq', 'Bsmt_Liveability_log',
    'Has_Basement', 'Winter_Resilience', 'Thermal_Efficiency',
    'Winter_x_Geo', 'Bsmt_Live_x_Geo',
]

# ============================================================
# Linear Model Guard: RobustScaler + clipping [10.5, 13.5]
# ============================================================
LOG_CLIP_LO = 10.5
LOG_CLIP_HI = 13.5

class RobustGuardPipeline:
    """Linear model: RobustScaler + output guard."""
    def __init__(self, model, selected_features=None):
        self.preprocess = HousePricesPreprocessor(selected_features=selected_features)
        self.model = model
        self.scaler_ = None
        self.label_maps_ = {}
    def fit(self, X, y):
        X_t = self.preprocess.fit_transform(X, y)
        X_t = self._le_fit(X_t)
        X_val = X_t.values.astype(np.float64)
        self.scaler_ = RobustScaler()
        X_val = self.scaler_.fit_transform(X_val)
        self.model.fit(X_val, y)
        return self
    def predict(self, X):
        X_t = self.preprocess.transform(X)
        X_t = self._le_transform(X_t)
        X_val = X_t.values.astype(np.float64)
        X_val = self.scaler_.transform(X_val)
        raw = self.model.predict(X_val)
        return np.clip(raw, LOG_CLIP_LO, LOG_CLIP_HI)
    def _le_fit(self, df):
        self.label_maps_ = {}
        for col in df.select_dtypes(include=['object']).columns:
            vals = sorted(df[col].unique())
            self.label_maps_[col] = {v: i for i, v in enumerate(vals)}
            df[col] = df[col].map(self.label_maps_[col]).fillna(-1).astype(int)
        return df
    def _le_transform(self, df):
        for col, m in self.label_maps_.items():
            if col in df.columns:
                df[col] = df[col].map(m).fillna(-1).astype(int)
        return df

# ============================================================
# Model Definitions & CV function
# ============================================================
MODEL_DEFS_V3 = {
    'Ridge_v3':      (Ridge(alpha=5.0), 'Linear', 'guard'),
    'ElasticNet_v3': (ElasticNet(alpha=0.0005, l1_ratio=0.7, max_iter=10000), 'Linear', 'guard'),
    'LightGBM':      (lgb.LGBMRegressor(
                          n_estimators=1000, learning_rate=0.05, max_depth=4,
                          num_leaves=15, min_child_samples=20, subsample=0.8,
                          colsample_bytree=0.8, random_state=SEED, verbose=-1),
                       'Boosting', 'sklearn'),
    'CatBoost':      (CatBoostRegressor(
                          iterations=1000, learning_rate=0.05, depth=6,
                          l2_leaf_reg=10, od_type='Iter', od_wait=50,
                          random_seed=SEED, verbose=0),
                       'Boosting', 'catboost'),
}

def run_cv_v3(name, model_template, paradigm, pipe_type, X, y, n_splits=N_SPLITS):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X))
    fold_scores = []
    last_pipeline = None
    for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
        model = copy.deepcopy(model_template)
        if pipe_type == 'guard':
            pipe = RobustGuardPipeline(model, selected_features=SELECTED_FEATURES)
        elif pipe_type == 'catboost':
            pipe = CatBoostPipeline(model, selected_features=SELECTED_FEATURES)
        else:
            pipe = SklearnPipeline(model, selected_features=SELECTED_FEATURES)
        pipe.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        preds = pipe.predict(X.iloc[va_idx])
        oof[va_idx] = preds
        rmse = np.sqrt(mean_squared_error(y.iloc[va_idx], preds))
        fold_scores.append(rmse)
        last_pipeline = pipe
    cv_mean = np.mean(fold_scores)
    cv_std = np.std(fold_scores)
    oof_rmse = np.sqrt(mean_squared_error(y, oof))
    print(f"  {name:16s} [{paradigm:8s}]  OOF={oof_rmse:.5f}  CV={cv_mean:.5f} +/- {cv_std:.5f}")
    return {'oof': oof, 'fold_scores': fold_scores, 'cv_mean': cv_mean,
            'cv_std': cv_std, 'oof_rmse': oof_rmse, 'paradigm': paradigm,
            'pipeline': last_pipeline}

print("Definitions loaded: add_iowa_survival_features, IOWA_FEATURES, RobustGuardPipeline, MODEL_DEFS_V3, run_cv_v3")

## Iowa Survival Model v4 — Generational Living Strategy

### 1. 世代別居住セグメント (Generational Segment)
- **Young_Transient**: 新築/リフォーム直後 × 小型 × Partial sale → 学生・若年層の一時拠点
- **Mid_Productive**: 中年齢 × ファミリーサイズ × Normal sale → 生産活動拠点
- **Senior_Settled**: 築古 × 1階建 × 長期保有 → 終の住処

### 2. 雪下ろしと維持管理 (Maintenance Burden)
- **Snow_Maintenance_Burden**: (Roof_Steepness × Roof_Area) / Accessibility_Score
- **Senior_Roof_Penalty**: 年配層 × 急勾配/複雑屋根 → 住み続けられないリスク

### 3. 社会的孤立回避 (Social Resilience)
- **Quiet_Density_Index**: 騒音回避 × 孤立回避の「静かな密集」評価
- **Life_Line_Proximity**: 幹線道路・鉄道近接 = 除雪優先/医療アクセス

### 4. 冬の生存指標アップグレード
- **Functional_Survival_Qual**: 断熱・暖房・基礎・屋根の実質生存品質
- **Thermal_Efficiency_v4**: 世代プロキシ加重の熱効率

### 5. Generational Adaptive Ensembler
- 世代プロキシに基づく動的重み: 若年→利便性, 年配→維持可能性+安心感

In [ ]:
# ============================================================
# Iowa Survival v4 — Feature Engineering
# ============================================================

def _sigmoid(x):
    """Numerically stable sigmoid."""
    return 1.0 / (1.0 + np.exp(-np.clip(x, -20, 20)))

def _gaussian(x, mu, sigma):
    """Gaussian bell curve [0, 1]."""
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2)

def add_iowa_v4_features(df):
    """
    V4 features: Generational Living + Maintenance Burden +
    Social Resilience + Functional Survival.
    Called BEFORE ordinal encoding (raw string values).
    """
    # ---- Pre-compute helpers ----
    house_age = df.get('HouseAge', df['YrSold'] - df.get('YearBuilt', df['YrSold']))
    remod_age = df.get('RemodAge', df['YrSold'] - df.get('YearRemodAdd', df['YrSold']))
    total_sf = (df.get('TotalBsmtSF', pd.Series(0, index=df.index)).fillna(0)
                + df.get('1stFlrSF', pd.Series(0, index=df.index)).fillna(0)
                + df.get('2ndFlrSF', pd.Series(0, index=df.index)).fillna(0))
    total_sf = total_sf.clip(lower=1)
    first_flr = df.get('1stFlrSF', pd.Series(0, index=df.index)).fillna(0)

    # ========================================================
    # 1. Generational Segment (soft probability scores)
    # ========================================================
    # --- Young_Transient: new/remodeled, small, partial/abnormal sale ---
    young_age = _sigmoid(-(house_age - 8) / 5)      # newer → higher
    young_remod = _sigmoid(-(remod_age - 5) / 3)     # recent remod → higher
    young_size = _sigmoid(-(total_sf - 1400) / 400)   # smaller → higher
    sale_cond = df.get('SaleCondition', pd.Series('Normal', index=df.index)).fillna('Normal')
    if sale_cond.dtype == object:
        young_sale = sale_cond.map({'Partial': 0.8, 'Abnorml': 0.5,
                                    'Normal': 0.3, 'Family': 0.2,
                                    'Alloca': 0.3, 'AdjLand': 0.3}).fillna(0.3)
    else:
        young_sale = 0.3
    df['Young_Transient'] = young_age * young_remod * 0.4 + young_size * 0.3 + young_sale * 0.3

    # --- Mid_Productive: moderate age, family-size, normal sale ---
    mid_age = _gaussian(house_age, mu=20, sigma=15)
    mid_size = _sigmoid((total_sf - 1200) / 500)
    if sale_cond.dtype == object:
        mid_sale = sale_cond.map({'Normal': 0.7, 'Partial': 0.5, 'Family': 0.8,
                                  'Abnorml': 0.3, 'Alloca': 0.4, 'AdjLand': 0.4}).fillna(0.5)
    else:
        mid_sale = 0.5
    df['Mid_Productive'] = mid_age * 0.4 + mid_size * 0.3 + mid_sale * 0.3

    # --- Senior_Settled: old, 1-story, long-held ---
    senior_age = _sigmoid((house_age - 30) / 10)
    senior_remod = _sigmoid((remod_age - 20) / 10)
    # 1-story indicator from MSSubClass or HouseStyle
    ms_sub = df.get('MSSubClass', pd.Series(20, index=df.index))
    one_story_classes = {20, 30, 40, 120}  # 1-story types
    is_one_story = ms_sub.isin(one_story_classes).astype(float)
    # Also check HouseStyle if available
    h_style = df.get('HouseStyle', pd.Series('', index=df.index))
    if h_style.dtype == object:
        is_one_story = np.maximum(is_one_story,
                                   h_style.isin(['1Story', '1.5Fin', '1.5Unf']).astype(float))
    df['Senior_Settled'] = senior_age * 0.35 + senior_remod * 0.25 + is_one_story * 0.25 + (1 - mid_size) * 0.15

    # ========================================================
    # 2. Snow Maintenance Burden
    # ========================================================
    # Roof steepness from RoofStyle
    roof_style = df.get('RoofStyle', pd.Series('Gable', index=df.index)).fillna('Gable')
    if roof_style.dtype == object:
        roof_steep = roof_style.map({
            'Flat': 0.0, 'Shed': 0.3, 'Hip': 0.4,
            'Gable': 0.7, 'Gambrel': 0.85, 'Mansard': 0.95
        }).fillna(0.5)
    else:
        roof_steep = 0.5

    # Roof area proxy: 1stFlrSF (footprint)
    roof_area_norm = first_flr / (first_flr.max() + 1e-9)

    # Accessibility score: PavedDrive + LotShape regularity + LandContour flatness
    paved = df.get('PavedDrive', pd.Series('Y', index=df.index)).fillna('Y')
    if paved.dtype == object:
        acc_paved = paved.map({'Y': 1.0, 'P': 0.5, 'N': 0.1}).fillna(0.5)
    else:
        acc_paved = paved / 2.0  # already ordinal encoded

    lot_shape = df.get('LotShape', pd.Series('Reg', index=df.index)).fillna('Reg')
    if lot_shape.dtype == object:
        acc_shape = lot_shape.map({'Reg': 1.0, 'IR1': 0.7, 'IR2': 0.4, 'IR3': 0.2}).fillna(0.5)
    else:
        acc_shape = 0.7

    contour = df.get('LandContour', pd.Series('Lvl', index=df.index)).fillna('Lvl')
    if contour.dtype == object:
        acc_contour = contour.map({'Lvl': 1.0, 'Bnk': 0.6, 'Low': 0.4, 'HLS': 0.3}).fillna(0.5)
    else:
        acc_contour = 0.7

    accessibility = (acc_paved * 0.4 + acc_shape * 0.3 + acc_contour * 0.3).clip(lower=0.1)

    df['Snow_Maintenance_Burden'] = (roof_steep * roof_area_norm) / accessibility

    # Senior_Roof_Penalty: penalty for seniors with hard-to-maintain roofs
    df['Senior_Roof_Penalty'] = df['Senior_Settled'] * df['Snow_Maintenance_Burden']

    # ========================================================
    # 3. Social Resilience
    # ========================================================
    # Quiet_Density_Index: Cul-de-sac = quiet, Inside = moderate,
    # Corner/FR2/FR3 = more exposed/noisy
    lot_config = df.get('LotConfig', pd.Series('Inside', index=df.index)).fillna('Inside')
    if lot_config.dtype == object:
        quiet_score = lot_config.map({
            'CulDSac': 1.0, 'Inside': 0.6, 'Corner': 0.4,
            'FR2': 0.3, 'FR3': 0.2
        }).fillna(0.5)
    else:
        quiet_score = 0.5

    # Density proxy: inverse of lot area (smaller lot = denser)
    lot_area = df.get('LotArea', pd.Series(8000, index=df.index)).fillna(8000)
    lot_area_log = np.log1p(lot_area)
    density = 1.0 - (lot_area_log - lot_area_log.min()) / (lot_area_log.max() - lot_area_log.min() + 1e-9)
    # "Quiet density" = quiet AND not too sparse
    optimal_density = _gaussian(density, mu=0.5, sigma=0.25)
    df['Quiet_Density_Index'] = quiet_score * 0.5 + optimal_density * 0.5

    # Life_Line_Proximity: arterial road / railroad = snow plow priority + medical access
    cond1 = df.get('Condition1', pd.Series('Norm', index=df.index)).fillna('Norm')
    cond2 = df.get('Condition2', pd.Series('Norm', index=df.index)).fillna('Norm')
    if cond1.dtype == object:
        # Artery/Feedr = near main roads (snow plow priority)
        # PosA/PosN = near park/amenity (walkable)
        road_score = cond1.map({
            'Artery': 0.9, 'Feedr': 0.7, 'PosA': 0.6, 'PosN': 0.5,
            'Norm': 0.4, 'RRAe': 0.5, 'RRAn': 0.5, 'RRNe': 0.3, 'RRNn': 0.3
        }).fillna(0.4)
        road_score2 = cond2.map({
            'Artery': 0.9, 'Feedr': 0.7, 'PosA': 0.6, 'PosN': 0.5,
            'Norm': 0.0, 'RRAe': 0.5, 'RRAn': 0.5, 'RRNe': 0.3, 'RRNn': 0.3
        }).fillna(0.0)
        life_road = np.maximum(road_score, road_score2)
    else:
        life_road = 0.4

    # Zoning: commercial/floating village = closer to services
    zoning = df.get('MSZoning', pd.Series('RL', index=df.index)).fillna('RL')
    if zoning.dtype == object:
        life_zone = zoning.map({
            'C (all)': 0.9, 'FV': 0.7, 'RM': 0.6, 'RH': 0.5, 'RL': 0.3
        }).fillna(0.4)
    else:
        life_zone = 0.4

    df['Life_Line_Proximity'] = life_road * 0.6 + life_zone * 0.4

    # ========================================================
    # 4. Functional Survival Quality (survival-only OverallQual)
    # ========================================================
    heating_map = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
    bsmt_q_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
    foundation_map = {'Wood': 1, 'Slab': 2, 'BrkTil': 3, 'CBlock': 4, 'Stone': 5, 'PConc': 6}
    exter_q_map = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

    heat_qc = df.get('HeatingQC', pd.Series(3, index=df.index))
    if heat_qc.dtype == object:
        heat_val = heat_qc.map(heating_map).fillna(3)
    else:
        heat_val = heat_qc.fillna(3)

    bsmt_q = df.get('BsmtQual', pd.Series(0, index=df.index))
    if bsmt_q.dtype == object:
        bsmt_val = bsmt_q.fillna('None').map(bsmt_q_map).fillna(0)
    else:
        bsmt_val = bsmt_q.fillna(0)

    found = df.get('Foundation', pd.Series('CBlock', index=df.index))
    if found.dtype == object:
        found_val = found.map(foundation_map).fillna(3)
    else:
        found_val = found.fillna(3)

    ext_q = df.get('ExterQual', pd.Series(3, index=df.index))
    if ext_q.dtype == object:
        ext_val = ext_q.map(exter_q_map).fillna(3)
    else:
        ext_val = ext_q.fillna(3)

    # Normalize each to [0, 1] and combine (survival-critical only)
    df['Functional_Survival_Qual'] = (
        (heat_val / 5.0) * 0.35 +       # Heating = most critical
        (bsmt_val / 5.0) * 0.20 +        # Basement quality
        (found_val / 6.0) * 0.20 +        # Foundation durability
        (ext_val / 5.0) * 0.15 +          # Exterior weatherproofing
        (roof_steep.clip(upper=0.5) / 0.5) * 0.10  # Moderate roof = better snow shedding
    )

    # ========================================================
    # Thermal_Efficiency v4: senior-weighted
    # ========================================================
    winter_r = df.get('Winter_Resilience', pd.Series(0, index=df.index)).fillna(0)
    base_thermal = np.where(total_sf > 0, winter_r / (total_sf / 1000.0 + 1e-9), 0.0)
    # Seniors value "warmth density" more
    senior_boost = 1.0 + df['Senior_Settled'] * 0.5  # up to +50% premium for seniors
    df['Thermal_Efficiency_v4'] = base_thermal * senior_boost

    # ========================================================
    # Interaction terms with KNN_Geo_Price
    # ========================================================
    if 'KNN_Geo_Price' in df.columns:
        geo = df['KNN_Geo_Price']
        df['Senior_x_Geo'] = df['Senior_Settled'] * geo
        df['QuietDensity_x_Geo'] = df['Quiet_Density_Index'] * geo
        df['LifeLine_x_Geo'] = df['Life_Line_Proximity'] * geo
        df['FuncSurvival_x_Geo'] = df['Functional_Survival_Qual'] * geo

    return df

V4_FEATURES = [
    'Young_Transient', 'Mid_Productive', 'Senior_Settled',
    'Snow_Maintenance_Burden', 'Senior_Roof_Penalty',
    'Quiet_Density_Index', 'Life_Line_Proximity',
    'Functional_Survival_Qual', 'Thermal_Efficiency_v4',
    'Senior_x_Geo', 'QuietDensity_x_Geo', 'LifeLine_x_Geo', 'FuncSurvival_x_Geo',
]

ALL_IOWA_FEATURES = IOWA_FEATURES + V4_FEATURES

print(f"V4 new features: {len(V4_FEATURES)}")
print(f"Total Iowa features (v3+v4): {len(ALL_IOWA_FEATURES)}")

In [ ]:
# ============================================================
# V4 Monkey-patch: add both v3 + v4 features
# ============================================================
def _patched_fit_transform_v4(self, X, y=None):
    df = X.copy()
    df = self._fill_missing(df)
    df = add_iowa_survival_features(df)   # v3
    df = add_iowa_v4_features(df)         # v4
    df = self._ordinal_encode(df)
    df = self._add_geo_features(df)
    df = self._add_domain_features(df)
    df = self._drop_redundant(df)
    df = self._fix_skewness(df)
    self.label_encoders_ = {}
    self.cat_columns_ = []
    for col in df.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        self.label_encoders_[col] = le
        self.cat_columns_.append(col)
    if self.selected_features is not None:
        keep = [c for c in self.selected_features if c in df.columns]
        for lf in ALL_IOWA_FEATURES:
            if lf in df.columns and lf not in keep:
                keep.append(lf)
        df = df[keep]
    self.feature_names_ = list(df.columns)
    self.cat_feature_indices_ = [i for i, c in enumerate(self.feature_names_) if c in self.cat_columns_]
    return df

def _patched_transform_v4(self, X):
    df = X.copy()
    df = self._fill_missing(df)
    df = add_iowa_survival_features(df)
    df = add_iowa_v4_features(df)
    df = self._ordinal_encode(df)
    df = self._add_geo_features(df)
    df = self._add_domain_features(df)
    df = self._drop_redundant(df)
    df = self._fix_skewness(df)
    for col, le in self.label_encoders_.items():
        if col in df.columns:
            known = set(le.classes_)
            df[col] = df[col].astype(str).apply(lambda x: x if x in known else le.classes_[0])
            df[col] = le.transform(df[col])
    if self.selected_features is not None:
        keep = [c for c in self.selected_features if c in df.columns]
        for lf in ALL_IOWA_FEATURES:
            if lf in df.columns and lf not in keep:
                keep.append(lf)
        df = df[keep]
    return df

HousePricesPreprocessor.fit_transform = _patched_fit_transform_v4
HousePricesPreprocessor.transform = _patched_transform_v4
print("Preprocessor patched: Iowa Survival v4 (Generational + Maintenance + Social)")

# ============================================================
# V4 Cross-Validation
# ============================================================
print("\n=== V4 Cross-Validation ===")
results_v4 = {}
for name, (model, paradigm, pipe_type) in MODEL_DEFS_V3.items():
    results_v4[name] = run_cv_v3(name, model, paradigm, pipe_type, X, y)
    gc.collect()

# TabNet with v4 features
preprocessor_v4 = HousePricesPreprocessor(selected_features=SELECTED_FEATURES)
X_processed_v4 = preprocessor_v4.fit_transform(X, y)
feature_names_v4 = preprocessor_v4.feature_names_
cat_idxs_v4 = preprocessor_v4.cat_feature_indices_
cat_dims_v4 = [int(X_processed_v4.iloc[:, i].nunique()) for i in cat_idxs_v4]
X_arr_v4 = X_processed_v4.values.astype(np.float64)
y_arr_v4 = y.values.reshape(-1, 1)

print(f"\nV4 features: {len(feature_names_v4)}")
v4_in = [f for f in feature_names_v4 if f in ALL_IOWA_FEATURES]
print(f"Iowa features in model: {len(v4_in)}")
print(f"  v3: {[f for f in v4_in if f in IOWA_FEATURES]}")
print(f"  v4: {[f for f in v4_in if f in V4_FEATURES]}")

results_v4['TabNet'] = run_tabnet_cv(X_arr_v4, y_arr_v4, feature_names_v4, cat_idxs_v4, cat_dims_v4)
gc.collect()

# V4 Model Ranking
print("\n=== V4 Model Ranking ===")
v4_sorted = sorted(results_v4.items(), key=lambda x: x[1]['oof_rmse'])
for i, (name, r) in enumerate(v4_sorted, 1):
    print(f"  {i}. {name:16s} [{r['paradigm']:8s}]  OOF={r['oof_rmse']:.5f}")

## Generational Adaptive Ensembler (v4)
- 世代プロキシに基づく3ティア重み: Young/Mid/Senior
- Young: 利便性重視 → TabNet (立地の非線形) 主軸
- Senior: 維持可能性重視 → CatBoost (屋根/カテゴリリスク) ブースト
- Huber Guard: 世代間の価値観乖離によるモデル不一致を補正

In [ ]:
from sklearn.linear_model import HuberRegressor

class GenerationalAdaptiveEnsembler:
    """
    V4 ensemble: weights adapt per-sample based on generational segment.
    - Young_Transient high → TabNet boost (location non-linearity)
    - Senior_Settled high → CatBoost boost (categorical risk: roof, maintenance)
    - Mid_Productive high → balanced
    - High price tier → CatBoost boost (same as v3)
    - Huber guard on extreme disagreement
    """
    def __init__(self, base_weights, young_tabnet_boost=1.3,
                 senior_catboost_boost=1.4, high_price_catboost_boost=1.2,
                 disagree_pct=90):
        self.base_weights = base_weights
        self.young_tabnet_boost = young_tabnet_boost
        self.senior_catboost_boost = senior_catboost_boost
        self.high_price_catboost_boost = high_price_catboost_boost
        self.disagree_pct = disagree_pct
        self.huber_ = None

    def fit(self, oof_preds, y_true, gen_segments=None):
        self.model_names_ = list(oof_preds.keys())

        # Disagreement detection
        if 'TabNet' in oof_preds and 'LightGBM' in oof_preds:
            self.disagree_ = np.abs(oof_preds['TabNet'] - oof_preds['LightGBM'])
        elif 'TabNet' in oof_preds and 'CatBoost' in oof_preds:
            self.disagree_ = np.abs(oof_preds['TabNet'] - oof_preds['CatBoost'])
        else:
            self.disagree_ = np.zeros(len(y_true))
        self.disagree_thresh_ = np.percentile(self.disagree_, self.disagree_pct)

        # Train Huber on disagreement samples
        high_mask = self.disagree_ > self.disagree_thresh_
        if high_mask.sum() > 10:
            X_stack = np.column_stack([oof_preds[n] for n in self.model_names_])
            # Add generational features to Huber if available
            if gen_segments is not None:
                X_stack = np.column_stack([X_stack, gen_segments])
            self.huber_ = HuberRegressor(epsilon=1.35)
            self.huber_.fit(X_stack[high_mask], y_true[high_mask])
            self._huber_has_gen = gen_segments is not None
            print(f"  Huber trained on {high_mask.sum()} disagreement samples"
                  f" (with gen features: {self._huber_has_gen})")
        else:
            self._huber_has_gen = False

        return self

    def predict(self, preds_dict, gen_young=None, gen_senior=None):
        n = len(next(iter(preds_dict.values())))

        # Base average for tier detection
        total_w = sum(self.base_weights[m] for m in self.model_names_)
        avg = sum(preds_dict[m] * self.base_weights[m] / total_w for m in self.model_names_)

        # Per-sample weight adjustment
        result = np.zeros(n)
        for i_sample in range(n):
            w = dict(self.base_weights)

            # High price tier → CatBoost boost
            if avg[i_sample] > 13.0 and 'CatBoost' in w:
                w['CatBoost'] *= self.high_price_catboost_boost

            # Generational adjustments
            if gen_young is not None and gen_senior is not None:
                young_s = gen_young[i_sample]
                senior_s = gen_senior[i_sample]

                # Young properties: TabNet sees location patterns better
                if 'TabNet' in w:
                    w['TabNet'] *= (1.0 + (self.young_tabnet_boost - 1.0) * young_s)

                # Senior properties: CatBoost handles categorical risk (roof, maintenance)
                if 'CatBoost' in w:
                    w['CatBoost'] *= (1.0 + (self.senior_catboost_boost - 1.0) * senior_s)

            # Normalize
            total = sum(w.values())
            for m in self.model_names_:
                result[i_sample] += preds_dict[m][i_sample] * w[m] / total

        # Huber override for extreme disagreement
        if self.huber_ is not None:
            if 'TabNet' in preds_dict and 'LightGBM' in preds_dict:
                disagree = np.abs(preds_dict['TabNet'] - preds_dict['LightGBM'])
            elif 'TabNet' in preds_dict and 'CatBoost' in preds_dict:
                disagree = np.abs(preds_dict['TabNet'] - preds_dict['CatBoost'])
            else:
                disagree = np.zeros(n)
            d_mask = disagree > self.disagree_thresh_
            if d_mask.any():
                X_stack = np.column_stack([preds_dict[m] for m in self.model_names_])
                if self._huber_has_gen and gen_young is not None and gen_senior is not None:
                    gen_seg = np.column_stack([gen_young, gen_senior])
                    X_stack = np.column_stack([X_stack, gen_seg])
                result[d_mask] = self.huber_.predict(X_stack[d_mask])
                print(f"  Huber override: {d_mask.sum()} samples")

        return result

# ============================================================
# Compute generational segments for OOF and test
# ============================================================
# We need raw (pre-ordinal) generational features
# Re-compute from raw X (before preprocessing)
_gen_prep = HousePricesPreprocessor(selected_features=None)
_X_gen = _gen_prep.fit_transform(X, y)
gen_young_train = _X_gen['Young_Transient'].values if 'Young_Transient' in _X_gen.columns else np.zeros(len(X))
gen_senior_train = _X_gen['Senior_Settled'].values if 'Senior_Settled' in _X_gen.columns else np.zeros(len(X))

_X_gen_test = _gen_prep.transform(X_test)
gen_young_test = _X_gen_test['Young_Transient'].values if 'Young_Transient' in _X_gen_test.columns else np.zeros(len(X_test))
gen_senior_test = _X_gen_test['Senior_Settled'].values if 'Senior_Settled' in _X_gen_test.columns else np.zeros(len(X_test))

print(f"Generational segments computed:")
print(f"  Young_Transient: train mean={gen_young_train.mean():.3f}, test mean={gen_young_test.mean():.3f}")
print(f"  Senior_Settled:  train mean={gen_senior_train.mean():.3f}, test mean={gen_senior_test.mean():.3f}")

# ============================================================
# OOF Validation: V4 Generational Ensemble
# ============================================================
oof_v4 = {name: r['oof'] for name, r in results_v4.items()}
gen_segments_train = np.column_stack([gen_young_train, gen_senior_train])

# --- Config A: TabNet + CatBoost (2-model) ---
ens_models_2 = ['TabNet', 'CatBoost']
oof_2 = {n: oof_v4[n] for n in ens_models_2}
ens_gen_2 = GenerationalAdaptiveEnsembler(
    base_weights={'TabNet': 0.81, 'CatBoost': 0.19},
    young_tabnet_boost=1.3, senior_catboost_boost=1.4
)
ens_gen_2.fit(oof_2, y.values, gen_segments=gen_segments_train)
oof_gen_2 = ens_gen_2.predict(oof_2, gen_young=gen_young_train, gen_senior=gen_senior_train)
rmse_gen_2 = np.sqrt(mean_squared_error(y.values, oof_gen_2))

# --- Config B: TabNet + LightGBM + CatBoost (3-model) ---
ens_models_3 = ['TabNet', 'LightGBM', 'CatBoost']
oof_3 = {n: oof_v4[n] for n in ens_models_3}
ens_gen_3 = GenerationalAdaptiveEnsembler(
    base_weights={'TabNet': 0.60, 'LightGBM': 0.15, 'CatBoost': 0.25},
    young_tabnet_boost=1.3, senior_catboost_boost=1.4
)
ens_gen_3.fit(oof_3, y.values, gen_segments=gen_segments_train)
oof_gen_3 = ens_gen_3.predict(oof_3, gen_young=gen_young_train, gen_senior=gen_senior_train)
rmse_gen_3 = np.sqrt(mean_squared_error(y.values, oof_gen_3))

print(f"\n=== V4 Generational Ensemble OOF Comparison ===")
print(f"  V4 Gen Adaptive 2-model:    {rmse_gen_2:.5f}")
print(f"  V4 Gen Adaptive 3-model:    {rmse_gen_3:.5f}")

## V4 Nelder-Mead + Submission
- 最適重み探索 → Generational Adaptive 適用 → submission_iowa_survival_v4.csv
- V3 vs V4 最終比較

In [ ]:
from scipy.optimize import minimize

# ============================================================
# Nelder-Mead: 2-model (TabNet + CatBoost)
# ============================================================
opt_models_v4 = ['TabNet', 'CatBoost']
oof_matrix_v4 = np.column_stack([oof_v4[n] for n in opt_models_v4])

def objective_v4_2(w_raw):
    w = np.exp(w_raw) / np.exp(w_raw).sum()
    blend = oof_matrix_v4 @ w
    return np.sqrt(mean_squared_error(y.values, blend))

res_v4_2 = minimize(objective_v4_2, np.log([0.81, 0.19]), method='Nelder-Mead',
                    options={'maxiter': 5000, 'xatol': 1e-8, 'fatol': 1e-8})
w_v4_2 = np.exp(res_v4_2.x) / np.exp(res_v4_2.x).sum()

print("=== V4 Nelder-Mead (TabNet + CatBoost) ===")
for n, w in zip(opt_models_v4, w_v4_2):
    print(f"  {n:12s}: {w:.4f}")
print(f"  OOF RMSLE:  {res_v4_2.fun:.5f}")

# ============================================================
# Nelder-Mead: 3-model (TabNet + LightGBM + CatBoost)
# ============================================================
opt_models_v4_3 = ['TabNet', 'LightGBM', 'CatBoost']
oof_matrix_v4_3 = np.column_stack([oof_v4[n] for n in opt_models_v4_3])

def objective_v4_3(w_raw):
    w = np.exp(w_raw) / np.exp(w_raw).sum()
    blend = oof_matrix_v4_3 @ w
    return np.sqrt(mean_squared_error(y.values, blend))

res_v4_3 = minimize(objective_v4_3, np.log([0.6, 0.15, 0.25]), method='Nelder-Mead',
                    options={'maxiter': 5000, 'xatol': 1e-8, 'fatol': 1e-8})
w_v4_3 = np.exp(res_v4_3.x) / np.exp(res_v4_3.x).sum()

print(f"\n=== V4 Nelder-Mead (3-model) ===")
for n, w in zip(opt_models_v4_3, w_v4_3):
    print(f"  {n:12s}: {w:.4f}")
print(f"  OOF RMSLE:  {res_v4_3.fun:.5f}")

# Pick best NM config
if res_v4_2.fun <= res_v4_3.fun:
    final_models_v4 = opt_models_v4
    final_weights_v4 = {n: float(w) for n, w in zip(opt_models_v4, w_v4_2)}
    final_nm_rmse_v4 = res_v4_2.fun
    print(f"\n  -> Using 2-model config")
else:
    final_models_v4 = opt_models_v4_3
    final_weights_v4 = {n: float(w) for n, w in zip(opt_models_v4_3, w_v4_3)}
    final_nm_rmse_v4 = res_v4_3.fun
    print(f"\n  -> Using 3-model config")

# ============================================================
# Generational Adaptive on NM weights
# ============================================================
oof_final_v4_subset = {n: oof_v4[n] for n in final_models_v4}
ens_final_v4 = GenerationalAdaptiveEnsembler(
    base_weights=final_weights_v4,
    young_tabnet_boost=1.3, senior_catboost_boost=1.4,
    high_price_catboost_boost=1.2, disagree_pct=90
)
ens_final_v4.fit(oof_final_v4_subset, y.values, gen_segments=gen_segments_train)
oof_final_v4 = ens_final_v4.predict(
    oof_final_v4_subset, gen_young=gen_young_train, gen_senior=gen_senior_train)
final_oof_rmse_v4 = np.sqrt(mean_squared_error(y.values, oof_final_v4))
print(f"\n  NM + Generational Adaptive OOF: {final_oof_rmse_v4:.5f}")

# ============================================================
# Test predictions & submission
# ============================================================
print("\n=== Generating submission_iowa_survival_v4.csv ===")

test_preds_v4 = {}
for name, r in results_v4.items():
    if name == 'TabNet':
        num_idxs_v4 = [i for i in range(X_arr_v4.shape[1]) if i not in cat_idxs_v4]
        scaler_tab_v4 = StandardScaler()
        scaler_tab_v4.fit(X_arr_v4[:, num_idxs_v4])
        X_test_v4 = preprocessor_v4.transform(X_test)
        X_test_arr_v4 = X_test_v4.values.astype(np.float64)
        X_test_scaled_v4 = X_test_arr_v4.copy()
        X_test_scaled_v4[:, num_idxs_v4] = scaler_tab_v4.transform(X_test_scaled_v4[:, num_idxs_v4])
        test_preds_v4[name] = r['model'].predict(X_test_scaled_v4).flatten()
    else:
        test_preds_v4[name] = r['pipeline'].predict(X_test)

test_final_v4_subset = {n: test_preds_v4[n] for n in final_models_v4}
test_ensemble_v4 = ens_final_v4.predict(
    test_final_v4_subset, gen_young=gen_young_test, gen_senior=gen_senior_test)
test_final_v4 = np.clip(np.expm1(test_ensemble_v4), CLIP_MIN, CLIP_MAX)

sub_v4 = pd.DataFrame({'Id': test_ids, 'SalePrice': test_final_v4})
sub_path_v4 = f'{SUBMISSION_DIR}/submission_iowa_survival_v4.csv'
sub_v4.to_csv(sub_path_v4, index=False)
print(f"  -> {sub_path_v4}")
print(f"  Range: ${test_final_v4.min():,.0f} - ${test_final_v4.max():,.0f}")
print(f"  Mean: ${test_final_v4.mean():,.0f}, Median: ${np.median(test_final_v4):,.0f}")

# ============================================================
# FINAL V3 vs V4 SUMMARY
# ============================================================
print(f"\n" + "=" * 70)
print(f"  V4 FINAL SUMMARY")
print(f"=" * 70)
print(f"  V4 best single model:  {min(r['oof_rmse'] for r in results_v4.values()):.5f}")
print(f"  V4 NM + Gen Adaptive:  {final_oof_rmse_v4:.5f}")
print(f"\n  V4 Weights: {final_weights_v4}")
print(f"  V4 File: {sub_path_v4}")
print(f"=" * 70)
gc.collect()

## Multi-Seed OOF + 2-Layer Stacking
1. TabNet / LightGBM / CatBoost を **5シードで学習** → OOF平均
2. Base OOF + V4主要特徴量 → **HuberRegressor メタモデル**
3. Generational Adaptive アンサンブルとスコア比較 → 改善時のみ submission 出力

In [ ]:
import gc, copy

MULTI_SEEDS = [42, 123, 456]  # 3 seeds (CPU-friendly, 5seeds: add 789, 2025)

# ============================================================
# 1. Multi-Seed OOF Collection
# ============================================================
def run_multiseed_oof(X, y, X_test, seeds=MULTI_SEEDS):
    """
    Run TabNet / LightGBM / CatBoost with multiple seeds,
    collect OOF predictions and test predictions for each.
    Returns: oof_avg dict, test_avg dict, per-seed scores.
    """
    n_train, n_test = len(X), len(X_test)
    model_names = ['TabNet', 'LightGBM', 'CatBoost']

    # Accumulators
    oof_all = {m: np.zeros((n_train, len(seeds))) for m in model_names}
    test_all = {m: np.zeros((n_test, len(seeds))) for m in model_names}
    seed_scores = []

    for si, seed in enumerate(seeds):
        print(f"\n--- Seed {seed} ({si+1}/{len(seeds)}) ---")
        kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

        # --- LightGBM ---
        lgb_model = lgb.LGBMRegressor(
            n_estimators=1000, learning_rate=0.05, max_depth=4,
            num_leaves=15, min_child_samples=20, subsample=0.8,
            colsample_bytree=0.8, random_state=seed, verbose=-1)
        oof_lgb = np.zeros(n_train)
        test_lgb = np.zeros(n_test)
        for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
            pipe = SklearnPipeline(copy.deepcopy(lgb_model), selected_features=SELECTED_FEATURES)
            pipe.fit(X.iloc[tr_idx], y.iloc[tr_idx])
            oof_lgb[va_idx] = pipe.predict(X.iloc[va_idx])
            test_lgb += pipe.predict(X_test) / N_SPLITS
        oof_all['LightGBM'][:, si] = oof_lgb
        test_all['LightGBM'][:, si] = test_lgb
        rmse_lgb = np.sqrt(mean_squared_error(y, oof_lgb))
        print(f"  LightGBM  OOF={rmse_lgb:.5f}")

        # --- CatBoost ---
        cb_model = CatBoostRegressor(
            iterations=1000, learning_rate=0.05, depth=6,
            l2_leaf_reg=10, od_type='Iter', od_wait=50,
            random_seed=seed, verbose=0)
        oof_cb = np.zeros(n_train)
        test_cb = np.zeros(n_test)
        for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
            pipe = CatBoostPipeline(copy.deepcopy(cb_model), selected_features=SELECTED_FEATURES)
            pipe.fit(X.iloc[tr_idx], y.iloc[tr_idx])
            oof_cb[va_idx] = pipe.predict(X.iloc[va_idx])
            test_cb += pipe.predict(X_test) / N_SPLITS
        oof_all['CatBoost'][:, si] = oof_cb
        test_all['CatBoost'][:, si] = test_cb
        rmse_cb = np.sqrt(mean_squared_error(y, oof_cb))
        print(f"  CatBoost  OOF={rmse_cb:.5f}")

        # --- TabNet ---
        prep_tn = HousePricesPreprocessor(selected_features=SELECTED_FEATURES)
        X_proc = prep_tn.fit_transform(X, y)
        feat_names = prep_tn.feature_names_
        c_idxs = prep_tn.cat_feature_indices_
        c_dims = [int(X_proc.iloc[:, ci].nunique()) for ci in c_idxs]
        X_np = X_proc.values.astype(np.float64)
        y_np = y.values.reshape(-1, 1)
        num_idxs = [i for i in range(X_np.shape[1]) if i not in c_idxs]

        tabnet_params = dict(
            n_d=8, n_a=8, n_steps=3, gamma=1.5, lambda_sparse=1e-3,
            cat_idxs=c_idxs, cat_dims=c_dims, cat_emb_dim=1,
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=2e-2, weight_decay=1e-5),
            scheduler_fn=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts,
            scheduler_params=dict(T_0=50, T_mult=2, eta_min=1e-5),
            mask_type='entmax', seed=seed, verbose=0, device_name=DEVICE)
        fit_params = dict(max_epochs=100, patience=15, batch_size=128,
                          virtual_batch_size=64, num_workers=0, drop_last=False)

        oof_tn = np.zeros(n_train)
        test_tn_folds = np.zeros((n_test, N_SPLITS))
        last_scaler = None
        last_model = None
        for fold, (tr_idx, va_idx) in enumerate(kf.split(X_np)):
            scaler = StandardScaler()
            X_tr = X_np[tr_idx].copy()
            X_va = X_np[va_idx].copy()
            X_tr[:, num_idxs] = scaler.fit_transform(X_tr[:, num_idxs])
            X_va[:, num_idxs] = scaler.transform(X_va[:, num_idxs])
            model = TabNetRegressor(**{**tabnet_params, 'seed': seed + fold})
            model.fit(X_tr, y_np[tr_idx], eval_set=[(X_va, y_np[va_idx])],
                      eval_metric=['rmse'], **fit_params)
            oof_tn[va_idx] = model.predict(X_va).flatten()
            # Test prediction for this fold
            X_test_proc = prep_tn.transform(X_test).values.astype(np.float64)
            X_test_sc = X_test_proc.copy()
            X_test_sc[:, num_idxs] = scaler.transform(X_test_sc[:, num_idxs])
            test_tn_folds[:, fold] = model.predict(X_test_sc).flatten()
            last_scaler = scaler
            last_model = model
            gc.collect()

        oof_all['TabNet'][:, si] = oof_tn
        test_all['TabNet'][:, si] = test_tn_folds.mean(axis=1)
        rmse_tn = np.sqrt(mean_squared_error(y, oof_tn))
        print(f"  TabNet    OOF={rmse_tn:.5f}")

        seed_scores.append({'seed': seed, 'TabNet': rmse_tn, 'LightGBM': rmse_lgb, 'CatBoost': rmse_cb})
        gc.collect()

    # Average across seeds
    oof_avg = {m: oof_all[m].mean(axis=1) for m in model_names}
    test_avg = {m: test_all[m].mean(axis=1) for m in model_names}

    print(f"\n=== Multi-Seed Averaged OOF ===")
    for m in model_names:
        rmse = np.sqrt(mean_squared_error(y, oof_avg[m]))
        single = np.sqrt(mean_squared_error(y, oof_all[m][:, 0]))
        print(f"  {m:12s}  single(seed0)={single:.5f}  avg({len(seeds)}seeds)={rmse:.5f}  delta={rmse-single:+.5f}")

    return oof_avg, test_avg, oof_all, test_all, seed_scores

print("Starting Multi-Seed OOF collection...")
oof_avg, test_avg, oof_all, test_all, seed_scores = run_multiseed_oof(X, y, X_test)

In [ ]:
# ============================================================
# 2. Two-Layer Stacking with Meta Features
# ============================================================
from sklearn.linear_model import HuberRegressor

# --- Meta features: V4 generational + survival ---
META_FEATURE_NAMES = [
    'Senior_Settled', 'Young_Transient', 'Mid_Productive',
    'Snow_Maintenance_Burden', 'Senior_Roof_Penalty',
    'Quiet_Density_Index', 'Life_Line_Proximity',
    'Functional_Survival_Qual', 'Thermal_Efficiency_v4',
]

# Extract meta features from preprocessed data
_meta_prep = HousePricesPreprocessor(selected_features=None)
_X_meta = _meta_prep.fit_transform(X, y)
_X_meta_test = _meta_prep.transform(X_test)

meta_train = np.column_stack([
    _X_meta[f].values if f in _X_meta.columns else np.zeros(len(X))
    for f in META_FEATURE_NAMES
])
meta_test = np.column_stack([
    _X_meta_test[f].values if f in _X_meta_test.columns else np.zeros(len(X_test))
    for f in META_FEATURE_NAMES
])

print(f"Meta features: {len(META_FEATURE_NAMES)}")
print(f"  {META_FEATURE_NAMES}")

# --- Build stacking input: OOF predictions + meta features ---
model_names_stack = ['TabNet', 'LightGBM', 'CatBoost']
X_stack_train = np.column_stack([oof_avg[m] for m in model_names_stack] + [meta_train])
X_stack_test = np.column_stack([test_avg[m] for m in model_names_stack] + [meta_test])

stack_feature_names = model_names_stack + META_FEATURE_NAMES
print(f"Stacking input: {X_stack_train.shape[1]} features "
      f"({len(model_names_stack)} OOF + {len(META_FEATURE_NAMES)} meta)")

# --- OOF CV for meta model ---
kf_meta = KFold(n_splits=5, shuffle=True, random_state=42)
oof_stacked = np.zeros(len(X))
test_stacked_folds = np.zeros((len(X_test), 5))

for fold, (tr_idx, va_idx) in enumerate(kf_meta.split(X_stack_train)):
    meta_model = HuberRegressor(epsilon=1.35, max_iter=500)
    meta_model.fit(X_stack_train[tr_idx], y.values[tr_idx])
    oof_stacked[va_idx] = meta_model.predict(X_stack_train[va_idx])
    test_stacked_folds[:, fold] = meta_model.predict(X_stack_test)

test_stacked = test_stacked_folds.mean(axis=1)
rmse_stacked = np.sqrt(mean_squared_error(y.values, oof_stacked))

# --- Also fit final meta model on full train ---
meta_final = HuberRegressor(epsilon=1.35, max_iter=500)
meta_final.fit(X_stack_train, y.values)
test_stacked_full = meta_final.predict(X_stack_test)

# Meta model weights
print(f"\nMeta model (HuberRegressor) coefficients:")
for fname, coef in zip(stack_feature_names, meta_final.coef_):
    print(f"  {fname:30s}: {coef:+.5f}")

# ============================================================
# 3. Comparison: V4 Adaptive vs Stacking
# ============================================================
# Simple NM blend (multi-seed averaged)
oof_nm_ms = np.column_stack([oof_avg[n] for n in final_models_v4])
w_vec = np.array([final_weights_v4[n] for n in final_models_v4])

def obj_ms(w_raw):
    w = np.exp(w_raw) / np.exp(w_raw).sum()
    return np.sqrt(mean_squared_error(y.values, oof_nm_ms @ w))

res_ms = minimize(obj_ms, np.log(w_vec), method='Nelder-Mead',
                  options={'maxiter': 5000, 'xatol': 1e-8, 'fatol': 1e-8})
w_ms = np.exp(res_ms.x) / np.exp(res_ms.x).sum()
rmse_nm_ms = res_ms.fun

# Generational adaptive on multi-seed averaged
oof_avg_subset = {n: oof_avg[n] for n in final_models_v4}
ens_ms = GenerationalAdaptiveEnsembler(
    base_weights={n: float(w) for n, w in zip(final_models_v4, w_ms)},
    young_tabnet_boost=1.3, senior_catboost_boost=1.4,
    high_price_catboost_boost=1.2, disagree_pct=90
)
ens_ms.fit(oof_avg_subset, y.values, gen_segments=gen_segments_train)
oof_gen_ms = ens_ms.predict(oof_avg_subset, gen_young=gen_young_train, gen_senior=gen_senior_train)
rmse_gen_ms = np.sqrt(mean_squared_error(y.values, oof_gen_ms))

print(f"\n{'=' * 70}")
print(f"  FINAL COMPARISON")
print(f"{'=' * 70}")
print(f"  V4 single-seed Adaptive:   {final_oof_rmse_v4:.5f}")
print(f"  Multi-seed NM blend:       {rmse_nm_ms:.5f}")
print(f"  Multi-seed Gen Adaptive:   {rmse_gen_ms:.5f}")
print(f"  2-Layer Stacking (Huber):  {rmse_stacked:.5f}")
print(f"{'=' * 70}")

# ============================================================
# 4. Pick best & generate submission
# ============================================================
candidates = {
    'gen_adaptive_ms': (rmse_gen_ms, None),
    'stacked': (rmse_stacked, None),
}

best_name = min(candidates, key=lambda k: candidates[k][0])
best_rmse = candidates[best_name][0]

print(f"\n  Best method: {best_name} (OOF={best_rmse:.5f})")

# Generate test predictions for best method
if best_name == 'stacked':
    # Use OOF-CV averaged test predictions (more robust than full-train)
    test_final_best = np.clip(np.expm1(test_stacked), CLIP_MIN, CLIP_MAX)
    sub_name = 'submission_v4_stacked.csv'
    sub_desc = f'Iowa v4 Stacked: MultiSeed+Huber OOF={rmse_stacked:.5f}'
else:
    test_avg_subset = {n: test_avg[n] for n in final_models_v4}
    test_gen_ms = ens_ms.predict(test_avg_subset, gen_young=gen_young_test, gen_senior=gen_senior_test)
    test_final_best = np.clip(np.expm1(test_gen_ms), CLIP_MIN, CLIP_MAX)
    sub_name = 'submission_v4_gen_adaptive_ms.csv'
    sub_desc = f'Iowa v4 GenAdaptive MultiSeed OOF={rmse_gen_ms:.5f}'

# Always output stacking submission if it improves over single-seed adaptive
if rmse_stacked < final_oof_rmse_v4:
    test_stacked_clip = np.clip(np.expm1(test_stacked), CLIP_MIN, CLIP_MAX)
    sub_stacked = pd.DataFrame({'Id': test_ids, 'SalePrice': test_stacked_clip})
    sub_stacked.to_csv(f'{SUBMISSION_DIR}/submission_v4_stacked.csv', index=False)
    print(f"  -> {SUBMISSION_DIR}/submission_v4_stacked.csv")
    print(f"     Range: ${test_stacked_clip.min():,.0f} - ${test_stacked_clip.max():,.0f}")
else:
    print(f"  Stacking did NOT improve over V4 adaptive ({rmse_stacked:.5f} >= {final_oof_rmse_v4:.5f})")
    print(f"  submission_v4_stacked.csv NOT generated.")

# Always output gen adaptive multi-seed
test_avg_subset_all = {n: test_avg[n] for n in final_models_v4}
test_gen_ms_pred = ens_ms.predict(test_avg_subset_all, gen_young=gen_young_test, gen_senior=gen_senior_test)
test_gen_ms_clip = np.clip(np.expm1(test_gen_ms_pred), CLIP_MIN, CLIP_MAX)
sub_gen = pd.DataFrame({'Id': test_ids, 'SalePrice': test_gen_ms_clip})
sub_gen.to_csv(f'{SUBMISSION_DIR}/submission_v4_gen_adaptive_ms.csv', index=False)
print(f"  -> {SUBMISSION_DIR}/submission_v4_gen_adaptive_ms.csv")
print(f"     Range: ${test_gen_ms_clip.min():,.0f} - ${test_gen_ms_clip.max():,.0f}")
print(f"     Mean: ${test_gen_ms_clip.mean():,.0f}, Median: ${np.median(test_gen_ms_clip):,.0f}")

# Final submission = best method
FINAL_SUB_PATH = f'{SUBMISSION_DIR}/{sub_name}'
print(f"\n  FINAL SUBMISSION: {FINAL_SUB_PATH}")
gc.collect()

## 提出ファイルダウンロード / Drive 同期

In [ ]:
import glob

if IS_COLAB:
    # Google Drive に成果物を自動同期
    DRIVE_OUTPUT = '/content/drive/MyDrive/kaggle-output/house-prices'
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)
    os.makedirs(f'{DRIVE_OUTPUT}/submissions', exist_ok=True)
    os.makedirs(f'{DRIVE_OUTPUT}/figures', exist_ok=True)

    for f in glob.glob(f'{SUBMISSION_DIR}/submission_*.csv'):
        shutil.copy2(f, f'{DRIVE_OUTPUT}/submissions/')
    print(f"Submissions -> {DRIVE_OUTPUT}/submissions/")

    for f in glob.glob(f'{FIGURES_DIR}/*.png'):
        shutil.copy2(f, f'{DRIVE_OUTPUT}/figures/')
    print(f"Figures -> {DRIVE_OUTPUT}/figures/")

    # final_submission.csv as fixed path
    if os.path.exists(FINAL_SUB_PATH):
        shutil.copy2(FINAL_SUB_PATH, '/content/final_submission.csv')
        print(f"Final -> /content/final_submission.csv")

    try:
        from google.colab import files
        files.download(FINAL_SUB_PATH)
    except Exception:
        print("(VSCode remote: auto-download skipped)")
else:
    print(f"Local: submissions saved in {SUBMISSION_DIR}/")
    !ls -la {SUBMISSION_DIR}/submission_*.csv

## Kaggle 自動提出

In [ ]:
import requests, time, json
from pathlib import Path

# --- Kaggle トークン取得 ---
_kaggle_token = None
_kaggle_user = None

# 1. ~/.kaggle/kaggle.json
_kj = Path.home() / '.kaggle' / 'kaggle.json'
if _kj.exists():
    _kdata = json.loads(_kj.read_text())
    _kaggle_token = _kdata.get('key', '')
    _kaggle_user = _kdata.get('username', '')

# 2. Colab Secrets
if not _kaggle_token and IS_COLAB:
    try:
        from google.colab import userdata
        _kaggle_user = userdata.get('KAGGLE_USERNAME')
        _raw = userdata.get('KAGGLE_KEY')
        if _raw:
            _kaggle_token = _raw
    except Exception:
        pass

# 3. Google Drive
if not _kaggle_token and IS_COLAB:
    _dkj = Path('/content/drive/MyDrive/.kaggle/kaggle.json')
    if _dkj.exists():
        _kdata = json.loads(_dkj.read_text())
        _kaggle_token = _kdata.get('key', '')
        _kaggle_user = _kdata.get('username', '')

if not _kaggle_token:
    print("Kaggle token not found. Submit manually.")
else:
    COMPETITION = 'house-prices-advanced-regression-techniques'
    SUBMIT_FILE = FINAL_SUB_PATH
    SUBMIT_DESC = sub_desc

    _headers = {'Authorization': f'Bearer {_kaggle_token}', 'Content-Type': 'application/json'}
    _file_size = os.path.getsize(SUBMIT_FILE)

    try:
        print(f'Submitting: {SUBMIT_FILE} ({_file_size:,} bytes)')
        print(f'Description: {SUBMIT_DESC}')

        # Step 1: StartSubmissionUpload
        print('Step 1/3: Upload init...', end='', flush=True)
        r = requests.post(
            'https://api.kaggle.com/v1/competitions.CompetitionApiService/StartSubmissionUpload',
            headers=_headers,
            json={'competitionName': COMPETITION,
                  'fileName': Path(SUBMIT_FILE).name,
                  'contentLength': _file_size})
        if not r.ok:
            raise Exception(f'Step 1 failed: HTTP {r.status_code} {r.text[:500]}')
        _resp = r.json()
        print(' OK')
        _blob_token = _resp.get('token', _resp.get('blobToken', ''))
        _upload_url = _resp.get('createUrl', _resp.get('uploadUrl', ''))

        # Step 2: PUT to GCS
        print('Step 2/3: Uploading...', end='', flush=True)
        with open(SUBMIT_FILE, 'rb') as f:
            r = requests.put(_upload_url, headers={'Content-Type': 'text/csv'}, data=f)
        if not r.ok:
            raise Exception(f'Step 2 failed: HTTP {r.status_code}')
        print(' OK')

        # Step 3: CreateSubmission
        print('Step 3/3: Creating submission...', flush=True)
        r = requests.post(
            'https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission',
            headers=_headers,
            json={'competitionName': COMPETITION,
                  'blobToken': _blob_token,
                  'submissionDescription': SUBMIT_DESC})
        if not r.ok:
            # Retry with array format
            r = requests.post(
                'https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission',
                headers=_headers,
                json={'competitionName': COMPETITION,
                      'blobFileTokens': [_blob_token],
                      'submissionDescription': SUBMIT_DESC})
            if not r.ok:
                raise Exception(f'Step 3 failed: HTTP {r.status_code} {r.text[:500]}')
        print(' OK')

        # Wait for score (max 5 min)
        print(f'\nWaiting for score (max 300s)...', flush=True)
        _start = time.time()
        while time.time() - _start < 300:
            time.sleep(10)
            r = requests.post(
                'https://api.kaggle.com/v1/competitions.CompetitionApiService/ListSubmissions',
                headers=_headers,
                json={'competitionName': COMPETITION, 'pageSize': 1})
            r.raise_for_status()
            _subs = r.json().get('submissions', [])
            if _subs:
                _latest = _subs[0]
                _status = _latest.get('status', '')
                _score = _latest.get('publicScore', '')
                _elapsed = int(time.time() - _start)
                if _status == 'COMPLETE' and _score:
                    print(f'\n{"=" * 50}')
                    print(f'  Public Score: {_score}')
                    print(f'  File: {_latest.get("fileName", "")}')
                    print(f'  Desc: {_latest.get("description", "")}')
                    print(f'  ({_elapsed}s)')
                    print(f'{"=" * 50}')
                    break
                elif _status == 'ERROR':
                    print(f'\nSubmission error: {_latest}')
                    break
                else:
                    print(f'  ... {_elapsed}s (status: {_status})', flush=True)
        else:
            print(f'\nTimeout (300s). Check Kaggle web.')

    except Exception as e:
        print(f'\nAuto-submit failed: {e}')
        print(f'Manual submit: {SUBMIT_FILE}')
    finally:
        del _kaggle_token, _headers, _file_size

## SHAP Analysis (Iowa Survival v3 + v4)
- LightGBM / CatBoost の SHAP values で Iowa 特徴量の貢献度を可視化
- Summary plot + Iowa features (v3 + v4) のインパクト分析

In [ ]:
import shap

# ============================================================
# SHAP: LightGBM v4
# ============================================================
print("=== SHAP Analysis: LightGBM v4 ===\n")

lgb_pipe = results_v4['LightGBM']['pipeline']
prep_lgb = lgb_pipe.preprocess
X_shap_lgb = prep_lgb.fit_transform(X, y)
# Apply label encoding (same as SklearnPipeline._le_fit)
for col in X_shap_lgb.select_dtypes(include=['object']).columns:
    X_shap_lgb[col] = X_shap_lgb[col].astype('category').cat.codes
X_shap_lgb_arr = X_shap_lgb.values.astype(np.float64)

explainer_lgb = shap.TreeExplainer(lgb_pipe.model)
shap_values_lgb = explainer_lgb.shap_values(X_shap_lgb_arr)

# Summary plot
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

plt.sca(axes[0])
shap.summary_plot(shap_values_lgb, X_shap_lgb, plot_type="bar",
                  max_display=25, show=False)
axes[0].set_title("LightGBM v4 SHAP (bar)", fontsize=14)

plt.sca(axes[1])
shap.summary_plot(shap_values_lgb, X_shap_lgb, max_display=25, show=False)
axes[1].set_title("LightGBM v4 SHAP (beeswarm)", fontsize=14)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/shap_lgb_v4.png', dpi=150, bbox_inches='tight')
plt.show()

# ============================================================
# SHAP: CatBoost v4 (use CatBoost native API for cat_features)
# ============================================================
print("\n=== SHAP Analysis: CatBoost v4 ===\n")

cb_pipe = results_v4['CatBoost']['pipeline']
prep_cb = cb_pipe.preprocess
X_shap_cb = prep_cb.fit_transform(X, y)

# CatBoost requires Pool with integer cat columns, not float array
from catboost import Pool as CatPool
_cat_idx = prep_cb.cat_feature_indices_
X_shap_cb_pool = X_shap_cb.copy()
for ci in _cat_idx:
    col = X_shap_cb_pool.columns[ci]
    X_shap_cb_pool[col] = X_shap_cb_pool[col].astype(int)
cb_pool = CatPool(X_shap_cb_pool, cat_features=_cat_idx)
_shap_raw = cb_pipe.model.get_feature_importance(data=cb_pool, type='ShapValues')
shap_values_cb = _shap_raw[:, :-1]  # last column is base value

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

plt.sca(axes[0])
shap.summary_plot(shap_values_cb, X_shap_cb, plot_type="bar",
                  max_display=25, show=False)
axes[0].set_title("CatBoost v4 SHAP (bar)", fontsize=14)

plt.sca(axes[1])
shap.summary_plot(shap_values_cb, X_shap_cb, max_display=25, show=False)
axes[1].set_title("CatBoost v4 SHAP (beeswarm)", fontsize=14)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/shap_cb_v4.png', dpi=150, bbox_inches='tight')
plt.show()

# ============================================================
# Iowa Features Impact Report
# ============================================================
print("\n" + "=" * 70)
print("  SHAP: IOWA FEATURES IMPACT REPORT (v3 + v4)")
print("=" * 70)

shap_imp_lgb = pd.Series(np.abs(shap_values_lgb).mean(axis=0), index=prep_lgb.feature_names_)
shap_imp_cb = pd.Series(np.abs(shap_values_cb).mean(axis=0), index=prep_cb.feature_names_)

# Normalize
shap_imp_lgb_norm = shap_imp_lgb / (shap_imp_lgb.max() + 1e-9)
shap_imp_cb_norm = shap_imp_cb / (shap_imp_cb.max() + 1e-9)

# Ranking
rank_lgb = shap_imp_lgb.rank(ascending=False).astype(int)
rank_cb = shap_imp_cb.rank(ascending=False).astype(int)

print(f"\n  {'Feature':30s}  {'LGB SHAP':>10s}  {'Rank':>5s}  {'CB SHAP':>10s}  {'Rank':>5s}")
print(f"  {'-'*30}  {'-'*10}  {'-'*5}  {'-'*10}  {'-'*5}")

# Show key existing features first
key_features = ['OverallQual', 'TotalSF', 'KNN_Geo_Price', 'HouseAge', 'TotalBath']
for feat in key_features:
    lgb_v = shap_imp_lgb_norm.get(feat, 0)
    lgb_r = rank_lgb.get(feat, '-')
    cb_v = shap_imp_cb_norm.get(feat, 0)
    cb_r = rank_cb.get(feat, '-')
    print(f"  {feat:30s}  {lgb_v:10.4f}  {lgb_r:>5}  {cb_v:10.4f}  {cb_r:>5}")

print()
# Iowa features
for feat in ALL_IOWA_FEATURES:
    lgb_v = shap_imp_lgb_norm.get(feat, 0)
    lgb_r = rank_lgb.get(feat, '-')
    cb_v = shap_imp_cb_norm.get(feat, 0)
    cb_r = rank_cb.get(feat, '-')
    tag = ' *** IOWA'
    print(f"  {feat:30s}  {lgb_v:10.4f}  {lgb_r:>5}  {cb_v:10.4f}  {cb_r:>5}{tag}")

# Total Iowa contribution
iowa_in_lgb = [f for f in ALL_IOWA_FEATURES if f in shap_imp_lgb.index]
iowa_in_cb = [f for f in ALL_IOWA_FEATURES if f in shap_imp_cb.index]
total_lgb = shap_imp_lgb[iowa_in_lgb].sum() / shap_imp_lgb.sum() * 100
total_cb = shap_imp_cb[iowa_in_cb].sum() / shap_imp_cb.sum() * 100

print(f"\n  Iowa features total SHAP contribution:")
print(f"    LightGBM: {total_lgb:.1f}% of total |SHAP|")
print(f"    CatBoost: {total_cb:.1f}% of total |SHAP|")
print("=" * 70)

gc.collect()